# Phase 2 — FinBERT Sentiment Ablation

## Goal

Test whether FinBERT's **3-class sentiment output** is enough for stock prediction, instead of using the full 768-D FinBERT embedding.

We compare:

| Experiment | Input |
|---|---|
| A | Price |
| B | Price + Fundamentals |
| C | Price + FinBERT 3-D sentiment |
| D | Price + Fundamentals + FinBERT 3-D sentiment |

Every experiment uses the same 5-day LSTM, chronological split, optimizer, and early stopping.

The key comparison is:

**3-D FinBERT sentiment vs 768-D FinBERT embedding.**



## 1. Setup

In [ ]:
import os, sys, glob, json, random, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score

REPO_ROOT="/content/capstone"
REPO_URL="https://github.com/AdityaMelkote3004/capstone.git"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git","clone",REPO_URL,REPO_ROOT],check=True)
else:
    subprocess.run(["git","-C",REPO_ROOT,"pull","--ff-only"],check=False)

sys.path.insert(0,REPO_ROOT)
!pip -q install transformers pyarrow scikit-learn

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED=42

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed()
print("PyTorch:",torch.__version__)
print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))


## 2. Load and Clean Dataset

In [ ]:
parquets=sorted(glob.glob(os.path.join(REPO_ROOT,"**","*.parquet"),recursive=True))
preferred=os.path.join(REPO_ROOT,"dataset","stocknet_final_modeling_set.parquet")
candidates=[p for p in parquets if "phase2" not in p.lower()]

if os.path.exists(preferred):
    BASE_PARQUET=preferred
elif candidates:
    BASE_PARQUET=candidates[0]
else:
    raise FileNotFoundError("Base modeling parquet not found.")

df=pd.read_parquet(BASE_PARQUET)
df["Date"]=pd.to_datetime(df["Date"])
df=df.sort_values(["Ticker","Date"]).reset_index(drop=True)

PRICE_FEATURES=[
    "Return","RSI_14","MACD","MACD_Signal","MACD_Hist",
    "Volatility_5","Volatility_20","Price_MA5_Ratio",
    "Price_MA10_Ratio","Price_MA20_Ratio","Volume_Change",
    "HL_Spread","MA_5","MA_10"
]

FUNDAMENTAL_FEATURES=[
    "Revenue","NetIncome","TotalAssets","TotalLiabilities",
    "StockholdersEquity","EPS","Cash","ROA"
]

required=PRICE_FEATURES+FUNDAMENTAL_FEATURES+[
    "Target","Ticker","Date","Company_Texts"
]
missing=[c for c in required if c not in df.columns]
assert not missing,f"Missing columns: {missing}"

# Fix the infinity problem found in the earlier experiments.
for c in PRICE_FEATURES+FUNDAMENTAL_FEATURES:
    df[c]=df[c].replace([np.inf,-np.inf],np.nan)

for c in PRICE_FEATURES:
    df[c]=df[c].fillna(0.0)

for c in FUNDAMENTAL_FEATURES:
    df[c]=df.groupby("Ticker")[c].ffill().fillna(0.0)

print("Shape:",df.shape)
print("Tickers:",df["Ticker"].nunique())


## 3. Load FinBERT as a Sequence Classification Model

**Important:** this is deliberately different from our previous embedding experiment.

Previous:

```python
AutoModel(...)
→ last_hidden_state[:, 0, :]
→ 768 dimensions
```

Now:

```python
AutoModelForSequenceClassification(...)
→ logits
→ softmax
→ [positive, neutral, negative]
```

This uses FinBERT's actual financial sentiment classification head.


In [ ]:
MODEL_NAME="ProsusAI/finbert"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
finbert=AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
).to(DEVICE)
finbert.eval()

print("Number of labels:",finbert.config.num_labels)
print("Label mapping:",finbert.config.id2label)


## 4. Extract 3-D FinBERT Sentiment

Each input text produces:

```text
[positive_probability,
 neutral_probability,
 negative_probability]
```

### Important data assumption

If `Company_Texts` contains one combined text per company/day, this cell produces **one sentiment vector for that company/day**.

If your dataset stores individual tweets separately, aggregate their individual FinBERT probabilities by company/day before using them as daily features.

The code below supports the current `Company_Texts` representation directly.


In [ ]:
texts=df["Company_Texts"].fillna("").astype(str).tolist()

daily_sentiment=np.zeros((len(df),3),dtype=np.float32)
BATCH_SIZE=32

with torch.no_grad():
    for start in range(0,len(texts),BATCH_SIZE):
        batch=texts[start:start+BATCH_SIZE]

        valid=[i for i,t in enumerate(batch) if t.strip()]

        if valid:
            inputs=tokenizer(
                [batch[i] for i in valid],
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )
            inputs={k:v.to(DEVICE) for k,v in inputs.items()}

            logits=finbert(**inputs).logits
            probs=torch.softmax(logits,dim=-1).cpu().numpy().astype(np.float32)

            for j,local_i in enumerate(valid):
                daily_sentiment[start+local_i]=probs[j]

        if start%(BATCH_SIZE*20)==0:
            print(f"{min(start+BATCH_SIZE,len(texts))}/{len(texts)}")

print("Shape:",daily_sentiment.shape)
print("NaN:",np.isnan(daily_sentiment).sum())
print("Inf:",np.isinf(daily_sentiment).sum())
print("First 5 probability vectors:")
print(daily_sentiment[:5])
print("Probability sums:",daily_sentiment[:5].sum(axis=1))


## 5. Chronological Split + Train-Only Normalization

In [ ]:
TRAIN_END=pd.Timestamp("2015-03-31")
VAL_START=pd.Timestamp("2015-04-01")
VAL_END=pd.Timestamp("2015-07-31")
TEST_START=pd.Timestamp("2015-08-01")

train_mask=df["Date"]<=TRAIN_END
val_mask=(df["Date"]>=VAL_START)&(df["Date"]<=VAL_END)
test_mask=df["Date"]>=TEST_START

price_mean=df.loc[train_mask,PRICE_FEATURES].mean()
price_std=df.loc[train_mask,PRICE_FEATURES].std().replace(0,1).fillna(1)

fund_mean=df.loc[train_mask,FUNDAMENTAL_FEATURES].mean()
fund_std=df.loc[train_mask,FUNDAMENTAL_FEATURES].std().replace(0,1).fillna(1)

price_arr=((df[PRICE_FEATURES]-price_mean)/price_std).fillna(0).to_numpy(np.float32)
fund_arr=((df[FUNDAMENTAL_FEATURES]-fund_mean)/fund_std).fillna(0).to_numpy(np.float32)

# Sentiment probabilities already lie in [0,1].
sentiment_arr=daily_sentiment.astype(np.float32)

print("Train:",int(train_mask.sum()))
print("Val:  ",int(val_mask.sum()))
print("Test: ",int(test_mask.sum()))

for name,a in [
    ("Price",price_arr),
    ("Fundamentals",fund_arr),
    ("Sentiment",sentiment_arr)
]:
    print(name,"NaN:",np.isnan(a).sum(),"Inf:",np.isinf(a).sum())


## 6. Build Identical 5-Day Samples

In [ ]:
WINDOW=5
work=df.copy()
work["_row_id"]=np.arange(len(work))

def build_samples(mask):
    frame=work.loc[mask]
    samples=[]

    for ticker,g in frame.groupby("Ticker"):
        g=g.sort_values("Date").reset_index(drop=True)
        ids=g["_row_id"].to_numpy()

        for i in range(WINDOW,len(g)):
            rows=ids[i-WINDOW:i]

            samples.append({
                "price":price_arr[rows],
                "fund":fund_arr[rows],
                "sentiment":sentiment_arr[rows],
                "target":int(g.iloc[i]["Target"]),
                "ticker":ticker,
                "date":g.iloc[i]["Date"]
            })

    return samples

class SentimentDataset(Dataset):
    def __init__(self,samples):
        self.samples=samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self,i):
        s=self.samples[i]
        return {
            "price":torch.tensor(s["price"],dtype=torch.float32),
            "fund":torch.tensor(s["fund"],dtype=torch.float32),
            "sentiment":torch.tensor(s["sentiment"],dtype=torch.float32),
            "target":torch.tensor(s["target"],dtype=torch.long)
        }

train_ds=SentimentDataset(build_samples(train_mask))
val_ds=SentimentDataset(build_samples(val_mask))
test_ds=SentimentDataset(build_samples(test_mask))

train_loader=DataLoader(train_ds,batch_size=64,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=256,shuffle=False)
test_loader=DataLoader(test_ds,batch_size=256,shuffle=False)

print("Train samples:",len(train_ds))
print("Val samples:  ",len(val_ds))
print("Test samples: ",len(test_ds))


## 7. Common LSTM

Only the input features change. The downstream LSTM is identical for all four experiments.

```text
A: 14 features
B: 22 features
C: 17 features
D: 25 features
```


In [ ]:
FEATURE_DIMS={
    "price":14,
    "fund":8,
    "sentiment":3
}

EXPERIMENTS={
    "A_Price":["price"],
    "B_Price_Fundamentals":["price","fund"],
    "C_Price_FinBERT_Sentiment":["price","sentiment"],
    "D_Price_Fundamentals_FinBERT_Sentiment":["price","fund","sentiment"]
}

class SentimentLSTM(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.lstm=nn.LSTM(
            input_dim,64,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.classifier=nn.Sequential(
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32,2)
        )

    def forward(self,x):
        _,(h_n,_)=self.lstm(x)
        return self.classifier(h_n[-1])

def make_input(batch,modalities):
    return torch.cat(
        [batch[m].to(DEVICE) for m in modalities],
        dim=-1
    )

def evaluate(model,loader,modalities):
    model.eval()
    y_true=[];y_pred=[];y_prob=[]

    with torch.no_grad():
        for batch in loader:
            logits=model(make_input(batch,modalities))

            y_true.extend(batch["target"].numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())
            y_prob.extend(torch.softmax(logits,1)[:,1].cpu().numpy())

    y_true=np.asarray(y_true)
    y_pred=np.asarray(y_pred)
    y_prob=np.asarray(y_prob)

    try:
        auc=roc_auc_score(y_true,y_prob)
    except ValueError:
        auc=0.5

    return {
        "accuracy":accuracy_score(y_true,y_pred),
        "f1":f1_score(y_true,y_pred,zero_division=0),
        "mcc":matthews_corrcoef(y_true,y_pred),
        "auc":auc
    }


## 8. Train the Four Experiments

In [ ]:
def train_experiment(name,modalities,epochs=40,patience=7):
    print("\n"+"="*80)
    print(name)
    print("="*80)

    set_seed()

    input_dim=sum(FEATURE_DIMS[m] for m in modalities)
    model=SentimentLSTM(input_dim).to(DEVICE)

    print("Input dimension:",input_dim)
    print("Parameters:",sum(p.numel() for p in model.parameters()))

    optimizer=torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    criterion=nn.CrossEntropyLoss()

    best_mcc=-1e9
    best_state=None
    wait=0

    for epoch in range(1,epochs+1):
        model.train()

        for batch in train_loader:
            optimizer.zero_grad()

            logits=model(make_input(batch,modalities))
            loss=criterion(
                logits,
                batch["target"].to(DEVICE)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            optimizer.step()

        val=evaluate(model,val_loader,modalities)

        if val["mcc"]>best_mcc:
            best_mcc=val["mcc"]
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            wait=0
        else:
            wait+=1

        if epoch==1 or epoch%5==0:
            print(
                f"Ep {epoch:3d} | "
                f"Val Acc={val['accuracy']:.3f} "
                f"F1={val['f1']:.3f} "
                f"MCC={val['mcc']:.4f} "
                f"AUC={val['auc']:.4f}"
            )

        if wait>=patience:
            print("Early stop at epoch",epoch)
            break

    model.load_state_dict(best_state)
    return model,evaluate(model,val_loader,modalities)

models={}
val_results={}

for name,modalities in EXPERIMENTS.items():
    models[name],val_results[name]=train_experiment(name,modalities)

print("\n"+"="*90)
print("VALIDATION RESULTS")
print("="*90)
print(f"{'Experiment':<45}{'Acc':>8}{'F1':>8}{'MCC':>8}{'AUC':>8}")

for name,m in val_results.items():
    print(
        f"{name:<45}"
        f"{m['accuracy']:>8.4f}"
        f"{m['f1']:>8.4f}"
        f"{m['mcc']:>8.4f}"
        f"{m['auc']:>8.4f}"
    )


## 9. Select Using Validation Only

In [ ]:
ranking=sorted(
    val_results.items(),
    key=lambda x:(x[1]["mcc"],x[1]["auc"]),
    reverse=True
)

for i,(name,m) in enumerate(ranking,1):
    print(
        f"{i}. {name} | "
        f"MCC={m['mcc']:.4f} | "
        f"AUC={m['auc']:.4f}"
    )

BEST_NAME=ranking[0][0]
print("\nSelected:",BEST_NAME)


## 10. Final Test Evaluation

In [ ]:
best_test=evaluate(
    models[BEST_NAME],
    test_loader,
    EXPERIMENTS[BEST_NAME]
)

print("="*80)
print("FINAL TEST RESULT")
print("="*80)
print("Configuration:",BEST_NAME)

for k,v in best_test.items():
    print(f"{k.upper():>10}: {v:.4f}")


## 11. Compare Against Previous 768-D FinBERT Results

Previous Phase 2A LSTM results:

| Representation | Accuracy | F1 | MCC | AUC |
|---|---:|---:|---:|---:|
| Price + Company FinBERT (768-D) | 0.5248 | 0.4721 | 0.0459 | 0.5304 |
| Price + Fundamentals + Company FinBERT (768-D) | 0.4870 | 0.6550 | 0.0000 | 0.5270 |

The new experiment tests whether **3-D financial sentiment** is enough to match or beat those richer representations.


In [ ]:
sentiment_results=pd.DataFrame([
    [name,m["accuracy"],m["f1"],m["mcc"],m["auc"]]
    for name,m in val_results.items()
],columns=[
    "Experiment","Val_Accuracy","Val_F1","Val_MCC","Val_AUC"
]).sort_values(
    ["Val_MCC","Val_AUC"],
    ascending=False
)

sentiment_results


## 12. Save Results + Generate Markdown

In [ ]:
RESULT_DIR=os.path.join(
    REPO_ROOT,
    "results",
    "phase2_finbert_sentiment"
)
os.makedirs(RESULT_DIR,exist_ok=True)

sentiment_results.to_csv(
    os.path.join(RESULT_DIR,"validation_results.csv"),
    index=False
)

results={
    "phase":"Phase 2",
    "experiment":"FinBERT Sentiment Ablation",
    "sentiment_dimension":3,
    "sentiment_labels":["positive","neutral","negative"],
    "window_size":WINDOW,
    "selection_rule":"Validation MCC, then validation AUC",
    "selected_configuration":BEST_NAME,
    "validation_results":val_results,
    "final_test_result":best_test,
    "previous_768d_lstm_reference":{
        "Price + Company FinBERT":{
            "accuracy":0.5248,
            "f1":0.4721,
            "mcc":0.0459,
            "auc":0.5304
        },
        "Price + Fundamentals + Company FinBERT":{
            "accuracy":0.4870,
            "f1":0.6550,
            "mcc":0.0000,
            "auc":0.5270
        }
    }
}

with open(os.path.join(RESULT_DIR,"results.json"),"w") as f:
    json.dump(results,f,indent=2)

report=[
"# Phase 2 — FinBERT Sentiment Ablation",
"",
"## Objective",
"",
"Test whether FinBERT's 3-class financial sentiment output is sufficient for stock prediction compared with the previously used 768-D FinBERT representation.",
"",
"## Representation",
"",
"Each text input is passed through ProsusAI/FinBERT's sequence-classification model. The classification logits are converted with softmax into three probabilities:",
"",
"- Positive",
"- Neutral",
"- Negative",
"",
"These probabilities form a 3-D sentiment representation.",
"",
"## Experiments",
"",
"| Experiment | Input |",
"|---|---|",
"| A | Price |",
"| B | Price + Fundamentals |",
"| C | Price + FinBERT Sentiment |",
"| D | Price + Fundamentals + FinBERT Sentiment |",
"",
"## Validation Results",
"",
"| Experiment | Accuracy | F1 | MCC | AUC |",
"|---|---:|---:|---:|---:|"
]

for _,r in sentiment_results.iterrows():
    report.append(
        f"| {r['Experiment']} | "
        f"{r['Val_Accuracy']:.4f} | "
        f"{r['Val_F1']:.4f} | "
        f"{r['Val_MCC']:.4f} | "
        f"{r['Val_AUC']:.4f} |"
    )

report += [
"",
"## Selected Configuration",
"",
f"**{BEST_NAME}** was selected using validation MCC, with AUC as the secondary criterion.",
"",
"## Final Test Result",
"",
"| Metric | Value |",
"|---|---:|"
]

for k,v in best_test.items():
    report.append(f"| {k.upper()} | {v:.4f} |")

report += [
"",
"## Previous 768-D FinBERT LSTM References",
"",
"| Feature Set | Accuracy | F1 | MCC | AUC |",
"|---|---:|---:|---:|---:|",
"| Price + Company FinBERT | 0.5248 | 0.4721 | 0.0459 | 0.5304 |",
"| Price + Fundamentals + Company FinBERT | 0.4870 | 0.6550 | 0.0000 | 0.5270 |",
"",
"## Interpretation",
"",
"If the 3-D sentiment representation performs similarly to the 768-D embedding, the simpler and more interpretable representation is preferable. If the 768-D embedding performs substantially better, the model is using information beyond sentiment.",
"",
"## Next Step",
"",
"Use this result to decide whether the centralized graph model should use simple FinBERT sentiment features or a richer text representation. Do not add graph structure or federated learning until the centralized baseline is established.",
""
]

report_path=os.path.join(
    RESULT_DIR,
    "phase2_finbert_sentiment.md"
)

Path(report_path).write_text(
    "\n".join(report),
    encoding="utf-8"
)

print("Results:",RESULT_DIR)
print("Markdown:",report_path)


## 13. Interpretation Guide

### If 3-D sentiment ≈ 768-D embedding

Then most of the useful text signal may simply be financial sentiment.

### If 768-D is clearly better

Then FinBERT is providing useful information beyond positive/neutral/negative sentiment.

### If both are weak

Then news sentiment/representation may not be the main source of predictive signal, strengthening the motivation to investigate relationships between companies with the graph model.

**Validation chooses the configuration. Test evaluates it.**
